# 05. Penugasan Armada

Menjawab pertanyaan #3: kapal mana (dari 35 armada) yang sebaiknya melayani tiap rute,
baik rute eksisting maupun rute baru yang diusulkan di analisis sebelumnya.

Penugasan dinilai terhadap empat batasan operasional:

1. **Draft vs alur pelabuhan** — `draft_m` kapal harus lebih kecil dari `max_draft_m`
   pelabuhan terdangkal di rute.
2. **Batas gelombang** — `max_operating_wave_m` kapal vs tinggi gelombang di perairan
   yang dilewati. Batasan ini ternyata yang paling menentukan (lihat temuan di bawah).
3. **Kapasitas waktu** — satu kapal tidak bisa di dua tempat sekaligus; jam-kapal yang
   dibutuhkan per minggu (`frekuensi x durasi`) tidak boleh melampaui ketersediaan.
4. **Status armada** — kapal `maintenance` tidak bisa ditugaskan.

Penugasan disambungkan ke hasil evaluasi rute (rute untung/rugi, kandidat ekspansi/hentikan)
dan rekomendasi rute baru, sehingga realokasi kapal punya arah yang jelas.

In [1]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
DATA = "../data/"
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (11, 3.6)

ports  = pd.read_csv(DATA+"ports.csv").set_index("port_id")
fleet  = pd.read_csv(DATA+"fleet.csv").set_index("ship_id")
routes = pd.read_csv(DATA+"routes_existing.csv").set_index("route_id")
orders = pd.read_csv(DATA+"orders_history_daily.csv", parse_dates=["date"])
prices = pd.read_csv(DATA+"route_prices.csv")
opex   = pd.read_csv(DATA+"route_opex_monthly.csv")
mob    = pd.read_csv(DATA+"mobility_daily.csv", parse_dates=["date"])

ww = pd.read_csv(DATA+"weather_wind_wave_daily.csv")
ww["date"] = pd.to_datetime(ww["date"], format="%d/%m/%Y")   # file ini DD/MM/YYYY
print("loaded |", len(ports), "pelabuhan |", len(fleet), "kapal |", len(routes), "rute eksisting")
print("status armada:", dict(fleet.status.value_counts()))

loaded | 19 pelabuhan | 35 kapal | 16 rute eksisting
status armada: {'active': np.int64(34), 'maintenance': np.int64(1)}


## 1. Fungsi kelayakan: draft, gelombang, kapasitas waktu

Tiga fungsi inti yang dipakai untuk menilai pasangan (kapal, rute).

**Batas gelombang -> pembatalan.** Untuk tiap perairan kita hitung fraksi hari dengan
gelombang melebihi batas operasi kapal. Angka ini langsung menjadi estimasi *rate
pembatalan* rute: kapal berbatas rendah di laut terbuka akan sering tidak berlayar.

In [2]:
# distribusi gelombang per perairan -> fungsi fraksi hari melebihi batas kapal
waves = {sea: g["wave_height_m"].values for sea, g in ww.groupby("sea_area")}
def frac_exceed(sea, lim):
    a = waves[sea]
    return float((a > lim).mean())

def route_seas(o, d):
    return ports.loc[o, "sea_area"], ports.loc[d, "sea_area"]

def draft_ok(ship, o, d):
    return fleet.loc[ship, "draft_m"] <= min(ports.loc[o, "max_draft_m"], ports.loc[d, "max_draft_m"])

def exp_cancel(ship, o, d):
    # estimasi rate pembatalan = fraksi hari gelombang > batas operasi kapal di perairan asal
    lim = fleet.loc[ship, "max_operating_wave_m"]
    so, sd = route_seas(o, d)
    return frac_exceed(so, lim), frac_exceed(sd, lim)

def util(freq_week, dur_hr):
    return freq_week * dur_hr / 168.0   # 168 jam per minggu

# kapasitas gelombang per kelas kapal (batas tipikal)
wlim = fleet.groupby("ship_type")["max_operating_wave_m"].median()
print("Batas gelombang tipikal per kelas kapal (m):")
print(wlim.to_string())

Batas gelombang tipikal per kelas kapal (m):
ship_type
Cargo/RoPax       3.0
Fast Ferry        1.8
Passenger Ship    3.5
RoRo Ferry        2.5


## 2. Kalibrasi: rate pembatalan ditentukan oleh kelas kapal vs gelombang

Sebelum menilai penugasan, kita uji dugaan bahwa pembatalan bukan acak melainkan
konsekuensi dari kelas kapal yang dipilih. Untuk tiap rute eksisting kita bandingkan
rate pembatalan aktual (dari riwayat pesanan) dengan estimasi `frac_exceed` di atas.

In [3]:
actual_cancel = orders.groupby("route_id")["cancelled"].mean()
chk = []
for rid, r in routes.iterrows():
    po, pd_ = exp_cancel(r.assigned_ship_id, r.origin_port_id, r.dest_port_id)
    chk.append([rid, r.assigned_ship_id, fleet.loc[r.assigned_ship_id,"ship_type"],
                fleet.loc[r.assigned_ship_id,"max_operating_wave_m"],
                round(po,3), round(actual_cancel[rid],3)])
chk = pd.DataFrame(chk, columns=["route","ship","ship_type","wave_lim","pred_cancel","actual_cancel"])
corr = chk.pred_cancel.corr(chk.actual_cancel)
mae  = (chk.pred_cancel - chk.actual_cancel).abs().mean()
print(f"Korelasi prediksi vs aktual: {corr:.3f} | MAE: {mae:.3f}")
chk

Korelasi prediksi vs aktual: 1.000 | MAE: 0.001


,route,ship,ship_type,wave_lim,pred_cancel,actual_cancel
0,R01,KM-02,Passenger Ship,3.5,0.015,0.015
1,R02,KM-31,Cargo/RoPax,3.0,0.047,0.052
2,R03,KM-15,RoRo Ferry,2.5,0.174,0.179
3,R04,KM-09,Passenger Ship,3.5,0.005,0.005
4,R05,KM-23,Fast Ferry,1.8,0.543,0.544
5,R06,KM-20,RoRo Ferry,2.5,0.161,0.161
6,R07,KM-16,RoRo Ferry,2.5,0.204,0.208
7,R08,KM-35,RoRo Ferry,2.5,0.161,0.161
8,R09,KM-06,Passenger Ship,3.5,0.015,0.015
9,R10,KM-08,Passenger Ship,3.5,0.000,0.000


Estimasi cocok hampir sempurna (korelasi ~1.0, MAE ~0.001). Jadi rate pembatalan tiap
rute praktis sama dengan peluang gelombang melampaui batas operasi kapal di perairan asal.

Konsekuensinya penting untuk penugasan: **pemilihan kelas kapal langsung mengontrol
keandalan berlayar.** Fast Ferry (batas ~1.8 m) di laut terbuka batal ~45-55% hari;
Passenger Ship (batas ~3.5 m) di perairan yang sama batal hanya ~0-2%. Penugasan kapal
bukan sekadar lolos/tidak, tapi pengungkit revenue lewat keandalan jadwal.

## 3. Audit penugasan saat ini

Untuk tiap rute eksisting: cek draft, estimasi pembatalan, utilisasi waktu, dan kinerja
permintaan (load factor, frekuensi sold-out). Plus P&L ringkas (revenue vs opex) agar
rekomendasi penugasan nyambung ke rute mana yang layak dipertahankan/diekspansi/dihentikan.

In [4]:
# P&L ringkas per rute - metode sama persis dgn evaluasi rute (notebook 03):
# pendapatan = total tiket per bulan (hari tak-berlayar otomatis 0) x tarif blended, dirata-rata.
def blend_price(g):
    p = g.set_index("ticket_class").price_idr
    return (.70*p["Ekonomi"]+.25*p["Bisnis"]+.05*p["Kabin"]) if "Kabin" in p.index else (.80*p["Ekonomi"]+.20*p["Bisnis"])
pb = prices.groupby("route_id").apply(blend_price).rename("pblend").reset_index()
om = orders.copy(); om["month"] = om.date.dt.strftime("%Y-%m")
mt = om.groupby(["route_id","month"]).tickets_sold.sum().rename("tickets").reset_index()
mm = mt.merge(pb, on="route_id").merge(opex[["route_id","month","total_opex_idr"]], on=["route_id","month"])
mm["rev"] = mm.tickets * mm.pblend
pl = mm.groupby("route_id").agg(rev_mo=("rev","mean"), opex_mo=("total_opex_idr","mean"))
pl["margin_pct"] = ((pl.rev_mo - pl.opex_mo) / pl.rev_mo * 100).round(1)   # margin / pendapatan

# metrik permintaan per rute (pada hari berlayar saja, konsisten dgn notebook 03)
dem = (orders[orders.trips>0].groupby("route_id")
       .agg(lf_mean=("load_factor","mean"),
            soldout=("load_factor", lambda s:(s>=0.999).mean()))).round(3)
dem["sail_rate"] = orders.groupby("route_id").trips.apply(lambda s:(s>0).mean()).round(3)

aud = routes.copy()
aud["draft_ok"]   = [draft_ok(r.assigned_ship_id, r.origin_port_id, r.dest_port_id) for _,r in aud.iterrows()]
aud["exp_cancel"] = [round(exp_cancel(r.assigned_ship_id, r.origin_port_id, r.dest_port_id)[0],3) for _,r in aud.iterrows()]
aud["util"]       = (aud.frequency_per_week * aud.scheduled_duration_hr / 168).round(2)
aud["ship_type"]  = fleet.loc[aud.assigned_ship_id, "ship_type"].values
aud = aud.join(dem).join(pl["margin_pct"])
cols = ["route_type","assigned_ship_id","ship_type","distance_nm","frequency_per_week",
        "draft_ok","exp_cancel","util","sail_rate","lf_mean","soldout","margin_pct"]
aud[cols]

,route_type,assigned_ship_id,ship_type,distance_nm,frequency_per_week,draft_ok,exp_cancel,util,sail_rate,lf_mean,soldout,margin_pct
route_id,,,,,,,,,,,,
R01,wajib,KM-02,Passenger Ship,18.3,84,True,0.015,0.95,0.985,0.896,0.314,76.2
R02,wajib,KM-31,Cargo/RoPax,4.2,96,True,0.047,0.57,0.948,0.917,0.377,-58.8
R03,wajib,KM-15,RoRo Ferry,31.5,14,True,0.174,0.27,0.821,0.311,0.000,-55.7
R04,rancangan,KM-09,Passenger Ship,277.2,7,True,0.005,0.82,0.995,0.786,0.089,76.7
R05,rancangan,KM-23,Fast Ferry,155.3,7,True,0.543,0.26,0.456,0.781,0.064,-19.3
R06,rancangan,KM-20,RoRo Ferry,494.7,5,True,0.161,1.09,0.595,0.642,0.021,44.2
R07,rancangan,KM-16,RoRo Ferry,208.6,6,True,0.204,0.64,0.693,0.720,0.038,47.2
R08,rancangan,KM-35,RoRo Ferry,147.3,4,True,0.161,0.35,0.490,0.573,0.007,-2.3
R09,rancangan,KM-06,Passenger Ship,202.5,3,True,0.015,0.25,0.411,0.593,0.018,31.9


### Pembacaan audit

- **Draft:** semua penugasan saat ini lolos (tidak ada kapal yang draft-nya melebihi alur
  pelabuhan). Jadi constraint pengikat ada di gelombang dan kapasitas waktu, bukan draft.
- **Pembatalan akibat salah kelas kapal:** R05, R12, R13, R14 memakai Fast Ferry di laut
  terbuka -> estimasi & aktual batal ~47-54% hari. Separuh jadwal hilang sebelum bicara
  permintaan. R05 khususnya sayang: saat berlayar sering hampir penuh, tapi keandalannya
  hancur karena kelas kapal.
- **Kapasitas waktu mepet/over:** R01 (util 0.95, wajib, sering sold-out) praktis butuh
  kapal kedua - satu kapal tak punya ruang untuk turnaround/perawatan. R06 util 1.09
  artinya satu kapal **tidak mungkin** memenuhi 5 keberangkatan/minggu di rute 495 nm.
- **Rugi + jarang sold-out + sering batal:** R13, R14, R15 rugi berat (margin -119% s/d
  -283%) -- pendapatan tak menutup biaya tetap karena frekuensi rendah dan separuh jadwal
  batal, sementara tak pernah penuh (tak ada permintaan tertekan) -> hentikan/realokasi.
  R12 rugi tipis tapi LF saat berlayar masih bagus -> rancang ulang (ganti kelas kapal untuk
  pangkas pembatalan) sebelum diputuskan hentikan. Kapal-kapalnya dibebaskan untuk realokasi.

In [5]:
# kelompokkan masalah
print("== Mismatch kelas kapal (estimasi batal > 20%) ==")
print(aud[aud.exp_cancel>0.20][["assigned_ship_id","ship_type","exp_cancel","lf_mean","margin_pct"]].to_string())
print()
print("== Kapasitas waktu mepet/over (util > 0.85) ==")
print(aud[aud.util>0.85][["assigned_ship_id","frequency_per_week","scheduled_duration_hr","util","route_type","soldout"]].to_string())
print()
# pool kapal menganggur (tak ditugaskan ke rute eksisting)
assigned = set(routes.assigned_ship_id)
idle = fleet[~fleet.index.isin(assigned)].copy()
print(f"== Armada menganggur: {len(idle)} kapal (1 maintenance) ==")
print(idle.groupby(["ship_type","status"]).size().to_string())

== Mismatch kelas kapal (estimasi batal > 20%) ==
         assigned_ship_id   ship_type  exp_cancel  lf_mean  margin_pct
route_id                                                              
R05                 KM-23  Fast Ferry       0.543    0.781       -19.3
R07                 KM-16  RoRo Ferry       0.204    0.720        47.2
R12                 KM-12  Fast Ferry       0.474    0.694       -12.9
R13                 KM-14  Fast Ferry       0.474    0.350      -186.8
R14                 KM-21  Fast Ferry       0.466    0.561      -282.8
R15                 KM-34  RoRo Ferry       0.250    0.291      -119.0
R16                 KM-32  RoRo Ferry       0.204    0.520        43.8

== Kapasitas waktu mepet/over (util > 0.85) ==
         assigned_ship_id  frequency_per_week  scheduled_duration_hr  util route_type  soldout
route_id                                                                                      
R01                 KM-02                  84                    1.9  0.9

## 4. Rencana realokasi

Logika: bebaskan kapal dari rute yang dihentikan/dirancang-ulang, tarik dari kolam
menganggur, lalu tugaskan ke (a) perbaikan keandalan rute bernilai, (b) penutup celah
kapasitas, dan (c) rute baru. Semua penugasan baru dicek ulang draft + gelombang + utilisasi.

Catatan kelas kapal: Fast Ferry (batas 1.8 m) secara struktural tak cocok untuk jaringan
ini -- setiap perairan melampaui 1.8 m pada ~40-59% hari. Kapal jenis ini sebaiknya jadi
cadangan / penyeberangan pendek di jendela tenang, bukan dipaksakan ke laut terbuka.

In [6]:
# kandidat kapal pengganti yang cocok gelombang (Passenger Ship, batas 3.5 m) & menganggur
pax_idle = idle[(idle.ship_type=="Passenger Ship") & (idle.status=="active")]
print("Passenger Ship menganggur (kandidat untuk rute laut terbuka):")
print(pax_idle[["ship_name","draft_m","pax_capacity","service_speed_knots","fuel_consumption_lph","daily_crew_cost_idr"]].to_string())

# rute baru dari analisis sebelumnya yang layak (skenario tengah positif): SMG-PJG, PRK-BBR
FACTOR = 1.272  # faktor koreksi jarak laut (median rute eksisting)
def hav_nm(a,b):
    la1,lo1,la2,lo2=map(np.radians,[ports.loc[a,"lat"],ports.loc[a,"lon"],ports.loc[b,"lat"],ports.loc[b,"lon"]])
    h=np.sin((la2-la1)/2)**2+np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2*6371*np.arcsin(np.sqrt(h))/1.852
for o,d in [("SMG","PJG"),("PRK","BBR")]:
    dist=hav_nm(o,d)*FACTOR
    print(f"{o}-{d}: jarak laut ~{dist:.0f} nm | perairan {route_seas(o,d)} | alur min {min(ports.loc[o,'max_draft_m'],ports.loc[d,'max_draft_m'])} m")

Passenger Ship menganggur (kandidat untuk rute laut terbuka):
                ship_name  draft_m  pax_capacity  service_speed_knots  fuel_consumption_lph  daily_crew_cost_idr
ship_id                                                                                                         
KM-03          KM Samudra      6.5           902                 15.8                   578              6000000
KM-05           KM Garuda      6.2          1260                 15.9                   426              4000000
KM-11    KM Anak Krakatau      6.6          1155                 16.3                   493             11000000
KM-24         KM Tenggiri      6.9           565                 15.9                   399              8000000
SMG-PJG: jarak laut ~404 nm | perairan ('Laut Jawa Tengah', 'Selat Sunda') | alur min 9.5 m
PRK-BBR: jarak laut ~288 nm | perairan ('Laut Jawa Barat', 'Selat Bangka') | alur min 8.5 m


In [7]:
# Susun rencana penugasan final (eksisting yang dipertahankan + perubahan + rute baru)
PILOT_FREQ = 3  # rute baru dimulai frekuensi rendah (pilot), sesuai rekomendasi analisis rute baru
plan = [
    # rute, o, d, tipe, kapal, freq, dur_hr, aksi
    ("R01","MRK","BKH","wajib","KM-02",84,1.9,"Pertahankan + tambah kapal ke-2 (KM-03): util 0.95 & sering sold-out"),
    ("R01b","MRK","BKH","wajib","KM-03",84,1.9,"Kapal ke-2 untuk R01 (ekspansi kapasitas rute padat)"),
    ("R02","KTP","GLM","wajib","KM-31",96,1.0,"Pertahankan: util 0.57, batal hanya 5%"),
    ("R03","PDB","BNO","wajib","KM-15",14,3.2,"Pertahankan (wajib); RoRo cukup, permintaan rendah"),
    ("R04","PRK","SMG","rancangan","KM-09",7,19.6,"Pertahankan: untung, batal ~0.5%, util 0.82"),
    ("R05","SMG","SBY","rancangan","KM-05",7,10.4,"GANTI Fast Ferry -> Passenger Ship: batal 54%->~2%, keandalan pulih"),
    ("R06","PRK","SBY","rancangan","KM-20",4,36.7,"Untung tapi util>1: kurangi 5->4/mgg agar muat 1 kapal (LF rendah, 5/mgg tak terjustifikasi)"),
    ("R07","SBY","BNO","rancangan","KM-16",6,17.8,"Pertahankan, pantau (RoRo, batal ~21%)"),
    ("R08","CBN","SMG","rancangan","KM-35",4,14.7,"Pertahankan, pantau"),
    ("R09","PJG","BBR","rancangan","KM-06",3,13.8,"Pertahankan (Passenger, batal ~1.5%)"),
    ("R10","BBR","BTM","rancangan","KM-08",3,18.7,"Pertahankan (batal ~0%)"),
    ("R11","BTM","TPI","rancangan","KM-10",21,3.5,"Pertahankan; naikkan freq dulu (util 0.44 ada ruang) sebelum tambah kapal"),
    ("R16","TBN","SBY","rancangan","KM-32",4,4.6,"Pertahankan, pantau"),
    # rute baru
    ("NEW SMG-PJG","SMG","PJG","baru","KM-11",PILOT_FREQ,404/15.0,"Rute baru (pilot): Passenger Ship utk laut terbuka, bukan Fast Ferry"),
    ("NEW PRK-BBR","PRK","BBR","baru","KM-24",PILOT_FREQ,288/15.0,"Rute baru (pilot): Passenger Ship, draft & gelombang aman"),
]
rows=[]
for rid,o,d,rt,ship,freq,dur,aksi in plan:
    po,pd_=exp_cancel(ship,o,d)
    rows.append(dict(rute=rid, kapal=ship, tipe=fleet.loc[ship,"ship_type"],
        draft_ok=draft_ok(ship,o,d), exp_cancel=round(po,3),
        util=round(util(freq,dur),2), freq=freq, route_type=rt, aksi=aksi))
P=pd.DataFrame(rows)
# cek keunikan kapal (1 kapal != 2 tempat)
dup=P.kapal[P.kapal.duplicated()].tolist()
print("Kapal dipakai >1 rute (harus kosong / sesuai rencana):", dup if dup else "tidak ada")
print("Semua draft lolos:", bool(P.draft_ok.all()))
P[["rute","kapal","tipe","route_type","draft_ok","exp_cancel","util","freq","aksi"]]

Kapal dipakai >1 rute (harus kosong / sesuai rencana): tidak ada
Semua draft lolos: True


,rute,kapal,tipe,route_type,draft_ok,exp_cancel,util,freq,aksi
0,R01,KM-02,Passenger Ship,wajib,True,0.015,0.95,84,Pertahankan + tambah kapal ke-2 (KM-03): util ...
1,R01b,KM-03,Passenger Ship,wajib,True,0.015,0.95,84,Kapal ke-2 untuk R01 (ekspansi kapasitas rute ...
2,R02,KM-31,Cargo/RoPax,wajib,True,0.047,0.57,96,"Pertahankan: util 0.57, batal hanya 5%"
3,R03,KM-15,RoRo Ferry,wajib,True,0.174,0.27,14,"Pertahankan (wajib); RoRo cukup, permintaan re..."
4,R04,KM-09,Passenger Ship,rancangan,True,0.005,0.82,7,"Pertahankan: untung, batal ~0.5%, util 0.82"
5,R05,KM-05,Passenger Ship,rancangan,True,0.020,0.43,7,GANTI Fast Ferry -> Passenger Ship: batal 54%-...
6,R06,KM-20,RoRo Ferry,rancangan,True,0.161,0.87,4,Untung tapi util>1: kurangi 5->4/mgg agar muat...
7,R07,KM-16,RoRo Ferry,rancangan,True,0.204,0.64,6,"Pertahankan, pantau (RoRo, batal ~21%)"
8,R08,KM-35,RoRo Ferry,rancangan,True,0.161,0.35,4,"Pertahankan, pantau"
9,R09,KM-06,Passenger Ship,rancangan,True,0.015,0.25,3,"Pertahankan (Passenger, batal ~1.5%)"


In [8]:
# Rute yang DIHENTIKAN / dirancang ulang -> kapal dibebaskan
stop = {
    "R12":"KM-12","R13":"KM-14","R14":"KM-21","R15":"KM-34",
}
freed = list(stop.values())
print("== Rute dihentikan/rancang-ulang (rugi + jarang sold-out + sering batal) ==")
for r,s in stop.items():
    a=aud.loc[r]
    print(f"  {r}: {s} ({fleet.loc[s,'ship_type']}) | margin {a.margin_pct}% | sold-out {a.soldout} | batal {a.exp_cancel} -> bebaskan kapal")
print("  Catatan: R13/R14/R15 rugi berat (margin -119%..-283%) -> hentikan/realokasi.")
print("  R12 rugi tipis (-13%) tapi LF saat berlayar bagus -> rancang ulang (ganti kelas kapal) sebelum hentikan.")
print()
# kapal yang dilepas dari penugasan lama (R05 fast ferry digantikan)
released = freed + ["KM-23"]
print("Kapal dibebaskan total:", released)
fast_idle = fleet.loc[[s for s in released if fleet.loc[s,'ship_type']=='Fast Ferry']]
print()
print("Fast Ferry yang dilepas -> cadangan / penyeberangan pendek (batas 1.8 m tak cocok laut terbuka):")
print(fast_idle[["ship_name","draft_m","max_operating_wave_m"]].to_string())

== Rute dihentikan/rancang-ulang (rugi + jarang sold-out + sering batal) ==
  R12: KM-12 (Fast Ferry) | margin -12.9% | sold-out 0.018 | batal 0.474 -> bebaskan kapal
  R13: KM-14 (Fast Ferry) | margin -186.8% | sold-out 0.0 | batal 0.474 -> bebaskan kapal
  R14: KM-21 (Fast Ferry) | margin -282.8% | sold-out 0.004 | batal 0.466 -> bebaskan kapal
  R15: KM-34 (RoRo Ferry) | margin -119.0% | sold-out 0.0 | batal 0.25 -> bebaskan kapal
  Catatan: R13/R14/R15 rugi berat (margin -119%..-283%) -> hentikan/realokasi.
  R12 rugi tipis (-13%) tapi LF saat berlayar bagus -> rancang ulang (ganti kelas kapal) sebelum hentikan.

Kapal dibebaskan total: ['KM-12', 'KM-14', 'KM-21', 'KM-34', 'KM-23']

Fast Ferry yang dilepas -> cadangan / penyeberangan pendek (batas 1.8 m tak cocok laut terbuka):
          ship_name  draft_m  max_operating_wave_m
ship_id                                           
KM-12     KM Wijaya      2.3                   1.8
KM-14    KM Pertiwi      2.8                   1.8
KM-

### Dampak yang bisa dikuantifikasi: keandalan R05

Penugasan ulang yang paling berdampak adalah **R05 SMG-SBY**: dari Fast Ferry (KM-23) ke
Passenger Ship (KM-05).

- **Keandalan berlayar** naik dari ~46% hari (batal 54%) menjadi ~98% hari -- sekitar 2.1x
  lebih banyak hari operasi.
- **Kapasitas per pelayaran** naik tajam (Fast Ferry ~239 kursi vs Passenger Ship ~1.260
  kursi), padahal saat berlayar rute ini sering hampir penuh -> ada permintaan tertekan yang
  selama ini hilang karena jadwal tak andal.

Catatan kejujuran: angka P&L bersih penugasan baru ini harus dihitung ulang -- Passenger
Ship lebih lambat dan boros bahan bakar daripada Fast Ferry, jadi biaya per pelayaran naik.
Yang ditegaskan di sini adalah keandalan dan kapasitas pulih; kelayakan finansial akhir
disambungkan ke model permintaan (notebook 02) dan model biaya (analisis rute baru), bukan
diklaim sebagai untung otomatis.

In [9]:
# ringkas keandalan R05 sebelum vs sesudah
before = exp_cancel("KM-23","SMG","SBY")[0]
after  = exp_cancel("KM-05","SMG","SBY")[0]
print(f"R05 SMG-SBY estimasi pembatalan: Fast Ferry KM-23 = {before:.2%}  ->  Passenger KM-05 = {after:.2%}")
print(f"Hari berlayar (per 365): {365*(1-before):.0f}  ->  {365*(1-after):.0f}  (~{(1-after)/(1-before):.1f}x)")
print(f"Kursi per pelayaran: {fleet.loc['KM-23','pax_capacity']}  ->  {fleet.loc['KM-05','pax_capacity']}")

R05 SMG-SBY estimasi pembatalan: Fast Ferry KM-23 = 54.29%  ->  Passenger KM-05 = 2.01%
Hari berlayar (per 365): 167  ->  358  (~2.1x)
Kursi per pelayaran: 239  ->  1260


## 5. Ringkasan penugasan armada

**Constraint pengikat.** Draft tidak mengikat (semua kapal muat di alur). Yang menentukan
adalah **gelombang vs kelas kapal** (mengontrol pembatalan, terbukti korelasi ~1.0 dengan
data) dan **kapasitas waktu** (util mingguan).

**Perubahan utama yang direkomendasikan:**

| Aksi | Rute | Dari -> Ke | Alasan |
|---|---|---|---|
| Ganti kelas kapal | R05 SMG-SBY | Fast Ferry -> Passenger Ship | Pembatalan 54% -> ~2%, keandalan & kapasitas pulih |
| Tambah kapal | R01 MRK-BKH (wajib) | 1 -> 2 kapal | Util 0.95 + sering sold-out (ekspansi rute padat) |
| Kurangi frekuensi | R06 PRK-SBY | 5 -> 4/minggu | Rute untung tapi util >1: 1 kapal tak bisa 5/mgg (LF rendah) |
| Hentikan/rancang ulang | R13, R14, R15 (R12 rancang ulang) | bebaskan 4 kapal | Rugi berat + jarang penuh + batal tinggi |
| Tugaskan rute baru | SMG-PJG, PRK-BBR | Passenger Ship (pilot) | Laut terbuka butuh kapal batas gelombang tinggi |

**Armada dibebaskan** (5 kapal: 4 Fast Ferry + 1 RoRo) menjadi cadangan untuk rotasi
perawatan dan penyeberangan pendek. Bersama kolam menganggur, jaringan punya kedalaman
cadangan yang sehat.

**Keterbatasan:** (1) penugasan ini berbasis frekuensi/durasi terjadwal -- konflik jadwal
harian detail (turnaround, sinkronisasi pelabuhan) perlu penjadwalan operasional lanjutan;
(2) batas gelombang diperlakukan statis per kapal -- model prediksi hari tidak-layar dari
cuaca (lihat opsi pemodelan) bisa menggantikan estimasi rata-rata dengan probabilitas
harian; (3) P&L penugasan baru perlu dihitung ulang dengan model permintaan + biaya, tidak
diklaim untung tanpa angka.